# AMTTrack v13 — MAX_BLIND cap + raw score analysis

Two changes from v12:

**1. KALMAN_MAX_BLIND now caps blind mode** (previously only widened search).
`blind = MIN_BLIND ≤ bfc ≤ MAX_BLIND`
Fixes d3_1:2hz (MaxRun=83→capped at 20) and pen_2hz (MaxRun=69→capped at 20)
where Kalman was predicting for 79+ frames and compounding velocity errors.

**2. `raw_score` signal saved** — the true network score on every frame,
including burst frames (where `pred_score` is forced to 0.0 for THOR suppression).
Used in the new Raw Score Analysis section to determine if burst-frame scores
differ between "Kalman helps" sequences (ball) and "Kalman hurts" sequences
(banana, d3), guiding a future score-based gate.

**Sections**
1. Prerequisites
2. Experiment config
3. Copy sequences
4. Run WITH motion model
5. Run WITHOUT motion model
6. Side-by-side metrics
7. Burst analysis
8. Raw score analysis (burst vs clean, MM vs No-MM)
9. Per-sequence signal plots

## Prerequisites

In [ ]:
%%capture
!git clone https://github.com/pofce/ATM.git FELT_SOT_Benchmark
%cd /content/FELT_SOT_Benchmark/AMTTrack_v2/

In [ ]:
%%capture
!pip install jpeg4py lmdb visdom

In [ ]:
import os, shutil, re
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from tqdm.notebook import tqdm
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%capture
!python tracking/create_default_local_file.py --workspace_dir . --data_dir ./data --save_dir ./output

In [ ]:
%%bash
set -e
SRC=/content/drive/MyDrive/Diploma_project/Datasets/AMTTrack/v2
mkdir -p pretrained_networks output/checkpoints/train/amttrack/fe108
cp "$SRC/pretrained_networks/OSTrack_ep0300.pth.tar" pretrained_networks/
cp "$SRC/AMTTrack_ep0098.pth.tar" output/checkpoints/train/amttrack/fe108/AMTTrack_ep0050.pth.tar

## Experiment Config

In [ ]:
# ── change these ───────────────────────────────────────────────────────────────
DATASET = 'fe108'

FLICKER_SEQUENCES = ['ball_1hz', 'ball_2hz', 'ball_4hz', 'ball_8hz', 'banana_1hz', 'banana_2hz', 'pen_1:2hz', 'pen_1hz', 'pen_2hz', 'pen_4hz', 'd3_1:2hz', 'd3_1hz', 'd3_2hz']
CONTROL_SEQUENCES = ['airplane_mul222', 'bike333', 'tower']
SEQUENCES         = FLICKER_SEQUENCES + CONTROL_SEQUENCES

FLICKER_DATA_ROOT = '/content/drive/MyDrive/Diploma_project/Datasets/Illumination_custom/seya/flicker'
CONTROL_DATA_ROOT = '/content/drive/MyDrive/Diploma_project/Datasets/FE108(v2)/test'
SEQ_DATA_ROOT     = {s: FLICKER_DATA_ROOT for s in FLICKER_SEQUENCES}
SEQ_DATA_ROOT.update({s: CONTROL_DATA_ROOT for s in CONTROL_SEQUENCES})

ADAPTIVE_MODALITY  = False
DISP_EMA_ALPHA     = 0.1
DVS_BG_LEVEL        = 255.0
DVS_BURST_THRESHOLD = 150.0
KALMAN_MIN_BLIND    = 4
KALMAN_MAX_BLIND    = 20

BASE        = '/content/FELT_SOT_Benchmark/AMTTrack_v2'
LOCAL_DATA  = f'{BASE}/data/FE108/test'
YAML_PATH   = f'{BASE}/experiments/amttrack/{DATASET}.yaml'
RESULTS_DIR_MM    = f'{BASE}/output/test/tracking_results/amttrack/{DATASET}_mm'
RESULTS_DIR_NO_MM = f'{BASE}/output/test/tracking_results/amttrack/{DATASET}_no_mm'

print(f'Sequences       : {SEQUENCES}')
print(f'BURST_THRESHOLD : {DVS_BURST_THRESHOLD}')
print(f'KALMAN_MIN_BLIND: {KALMAN_MIN_BLIND}')
print(f'KALMAN_MAX_BLIND: {KALMAN_MAX_BLIND}')

In [ ]:
# Copy sequences from Drive
for seq in tqdm(SEQUENCES, desc='Copying'):
    dst = os.path.join(LOCAL_DATA, seq)
    os.makedirs(dst, exist_ok=True)
    src = os.path.join(SEQ_DATA_ROOT[seq], seq)
    for sub in ('aps', 'dvs'):
        shutil.copytree(os.path.join(src, sub), os.path.join(dst, sub), dirs_exist_ok=True)
    shutil.copy2(os.path.join(src, 'groundtruth_rect.txt'), os.path.join(dst, 'groundtruth_rect.txt'))
print('Done.')

In [ ]:
def patch_yaml(motion_model: bool):
    _yaml = open(YAML_PATH).read()
    for key, val in [
        ('MOTION_MODEL',        motion_model),
        ('ADAPTIVE_MODALITY',   ADAPTIVE_MODALITY),
        ('DISP_EMA_ALPHA',      DISP_EMA_ALPHA),
        ('DVS_BG_LEVEL',        DVS_BG_LEVEL),
        ('DVS_BURST_THRESHOLD', DVS_BURST_THRESHOLD),
        ('KALMAN_MIN_BLIND',    KALMAN_MIN_BLIND),
        ('KALMAN_MAX_BLIND',    KALMAN_MAX_BLIND),
    ]:
        _yaml = re.sub(rf'({key}\s*:\s*)\S+', rf'\g<1>{val}', _yaml)
    open(YAML_PATH, 'w').write(_yaml)
    print(f'Patched yaml: MOTION_MODEL={motion_model}')

In [ ]:
# Write list.txt
seq_dirs = sorted([d for d in os.listdir(LOCAL_DATA) if os.path.isdir(os.path.join(LOCAL_DATA, d))])
with open(os.path.join(LOCAL_DATA, 'list.txt'), 'w') as f:
    f.write('\n'.join(seq_dirs))
print(f'Indexed {len(seq_dirs)} sequences: {seq_dirs}')

## Run Tracking — WITH Motion Model

In [ ]:
patch_yaml(motion_model=True)
if os.path.isdir(RESULTS_DIR_MM):
    shutil.rmtree(RESULTS_DIR_MM)
    print(f'Cleared: {RESULTS_DIR_MM}')

In [ ]:
%cd /content/FELT_SOT_Benchmark/AMTTrack_v2/
!python tracking/test.py \
    --tracker_name amttrack \
    --tracker_param {DATASET} \
    --dataset_name {DATASET} \
    --num_gpus 1
# move results to MM directory
import glob
base_results = f'{BASE}/output/test/tracking_results/amttrack/{DATASET}'
if os.path.isdir(base_results) and base_results != RESULTS_DIR_MM:
    shutil.copytree(base_results, RESULTS_DIR_MM, dirs_exist_ok=True)
    shutil.rmtree(base_results)
print(f'MM results saved → {RESULTS_DIR_MM}')

## Run Tracking — WITHOUT Motion Model

In [ ]:
patch_yaml(motion_model=False)
if os.path.isdir(RESULTS_DIR_NO_MM):
    shutil.rmtree(RESULTS_DIR_NO_MM)
    print(f'Cleared: {RESULTS_DIR_NO_MM}')

In [ ]:
%cd /content/FELT_SOT_Benchmark/AMTTrack_v2/
!python tracking/test.py \
    --tracker_name amttrack \
    --tracker_param {DATASET} \
    --dataset_name {DATASET} \
    --num_gpus 1
base_results = f'{BASE}/output/test/tracking_results/amttrack/{DATASET}'
if os.path.isdir(base_results) and base_results != RESULTS_DIR_NO_MM:
    shutil.copytree(base_results, RESULTS_DIR_NO_MM, dirs_exist_ok=True)
    shutil.rmtree(base_results)
print(f'No-MM results saved → {RESULTS_DIR_NO_MM}')

## Side-by-Side Metrics

In [ ]:
def calc_metrics(gt, pr):
    n = min(len(gt), len(pr))
    gt, pr = gt[:n], pr[:n]
    gt_cx = gt[:, 0] + gt[:, 2] / 2;  gt_cy = gt[:, 1] + gt[:, 3] / 2
    pr_cx = pr[:, 0] + pr[:, 2] / 2;  pr_cy = pr[:, 1] + pr[:, 3] / 2
    dist  = np.sqrt((gt_cx - pr_cx)**2 + (gt_cy - pr_cy)**2)
    precision = np.mean(dist <= 20)
    ix1 = np.maximum(gt[:, 0], pr[:, 0]); iy1 = np.maximum(gt[:, 1], pr[:, 1])
    ix2 = np.minimum(gt[:, 0]+gt[:, 2], pr[:, 0]+pr[:, 2])
    iy2 = np.minimum(gt[:, 1]+gt[:, 3], pr[:, 1]+pr[:, 3])
    inter = np.maximum(0, ix2-ix1) * np.maximum(0, iy2-iy1)
    union = gt[:, 2]*gt[:, 3] + pr[:, 2]*pr[:, 3] - inter
    iou   = inter / (union + 1e-10)
    return precision, np.mean(iou > 0.5), dist, iou

def load_results(results_dir):
    res = {}
    for seq in SEQUENCES:
        gt_path = os.path.join(SEQ_DATA_ROOT[seq], seq, 'groundtruth_rect.txt')
        pr_path = os.path.join(results_dir, f'{seq}.txt')
        if not os.path.isfile(pr_path):
            continue
        gt = np.loadtxt(gt_path, delimiter=',')
        pr = np.loadtxt(pr_path)
        prec, succ, dist, iou = calc_metrics(gt, pr)
        res[seq] = dict(precision=prec, success=succ, dist=dist, iou=iou, gt=gt, pr=pr)
    return res

results_mm    = load_results(RESULTS_DIR_MM)
results_no_mm = load_results(RESULTS_DIR_NO_MM)

print(f'  {"Sequence":<18} {"No-MM P":>8} {"MM P":>8} {"Delta":>8} {"No-MM S":>8} {"MM S":>8}')
print('  ' + '─'*68)
for seq in SEQUENCES:
    label = 'FLICKER' if seq in FLICKER_SEQUENCES else 'control'
    p_no  = results_no_mm[seq]['precision'] if seq in results_no_mm else float('nan')
    p_mm  = results_mm[seq]['precision']    if seq in results_mm    else float('nan')
    s_no  = results_no_mm[seq]['success']   if seq in results_no_mm else float('nan')
    s_mm  = results_mm[seq]['success']      if seq in results_mm    else float('nan')
    delta = p_mm - p_no
    marker = ' ▲' if delta > 0.005 else (' ▼' if delta < -0.005 else '  ')
    print(f'  [{label}] {seq:<18} {p_no:>8.4f} {p_mm:>8.4f} {delta:>+8.4f}{marker}  {s_no:>8.4f} {s_mm:>8.4f}')

## Burst Analysis (MM run)

In [ ]:
def load_sig(seq, key, results_dir=None):
    d = results_dir or RESULTS_DIR_MM
    path = os.path.join(d, f'{seq}_{key}.txt')
    if not os.path.isfile(path): return None
    arr = np.loadtxt(path)
    return arr if arr.ndim > 0 else arr.reshape(1)

print(f'  {"Sequence":<18} {"Burst%":>7} {"Kalman%":>8} {"MaxRun":>7} {"MedRun":>7}')
print('  ' + '─'*52)
for seq in SEQUENCES:
    burst = load_sig(seq, 'burst_detected')
    bfc   = load_sig(seq, 'burst_frame_count')
    if burst is None:
        print(f'  [MISSING] {seq}'); continue
    burst_pct  = burst.mean() * 100
    # Actual Kalman activation: MIN_BLIND <= bfc <= MAX_BLIND
    kalman_pct = ((bfc >= KALMAN_MIN_BLIND) & (bfc <= KALMAN_MAX_BLIND)).mean() * 100
    runs, run = [], 0
    for b in burst:
        if b > 0: run += 1
        else:
            if run > 0: runs.append(run); run = 0
    if run > 0: runs.append(run)
    max_run = max(runs) if runs else 0
    med_run = int(np.median(runs)) if runs else 0
    label = 'FLICKER' if seq in FLICKER_SEQUENCES else 'control'
    print(f'  [{label}] {seq:<18} {burst_pct:>6.1f}% {kalman_pct:>7.1f}% {max_run:>7} {med_run:>7}')

## Per-Sequence Signal Plots

Panel 1: raw_score (MM) vs raw_score (No-MM) — true score, not zeroed during burst
Panel 2: norm_displacement (MM)
Panel 3: dvs_activity + Kalman-active shading (green: MIN≤bfc≤MAX, gold: burst but inactive)

In [ ]:
print(f'  {"Sequence":<18} {"Clean μ":>8} {"Burst μ":>8} {"Drop":>8} {"No-MM P":>8} {"MM P":>8} {"Delta":>8}')
print('  ' + '─'*72)
for seq in SEQUENCES:
    raw_mm  = load_sig(seq, 'raw_score',      RESULTS_DIR_MM)
    burst   = load_sig(seq, 'burst_detected', RESULTS_DIR_MM)
    if raw_mm is None or burst is None:
        print(f'  [MISSING] {seq}'); continue
    n = min(len(raw_mm), len(burst))
    raw_mm, burst = raw_mm[:n], burst[:n]
    clean_mask = burst < 0.5
    burst_mask = burst > 0.5
    mean_clean = raw_mm[clean_mask].mean() if clean_mask.any() else float('nan')
    mean_burst = raw_mm[burst_mask].mean() if burst_mask.any() else float('nan')
    drop = mean_clean - mean_burst
    p_no = results_no_mm[seq]['precision'] if seq in results_no_mm else float('nan')
    p_mm = results_mm[seq]['precision']    if seq in results_mm    else float('nan')
    label = 'FLICKER' if seq in FLICKER_SEQUENCES else 'ctrl'
    marker = ' ▲' if (p_mm - p_no) > 0.005 else (' ▼' if (p_mm - p_no) < -0.005 else '  ')
    print(f'  [{label}] {seq:<18} {mean_clean:>8.3f} {mean_burst:>8.3f} {drop:>8.3f}  {p_no:>8.4f} {p_mm:>8.4f} {p_mm-p_no:>+8.4f}{marker}')

# Plot: burst score distributions for flicker sequences
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
helps  = [s for s in FLICKER_SEQUENCES if s in results_mm and s in results_no_mm
          and results_mm[s]['precision'] > results_no_mm[s]['precision'] + 0.01]
hurts  = [s for s in FLICKER_SEQUENCES if s in results_mm and s in results_no_mm
          and results_mm[s]['precision'] < results_no_mm[s]['precision'] - 0.01]

ax = axes[0]
ax.set_title('Raw score during BURST frames\n(green=MM helps, red=MM hurts)', fontsize=10)
for seq, color in [(s, 'green') for s in helps] + [(s, 'red') for s in hurts]:
    raw = load_sig(seq, 'raw_score', RESULTS_DIR_MM)
    bst = load_sig(seq, 'burst_detected', RESULTS_DIR_MM)
    if raw is None or bst is None: continue
    n = min(len(raw), len(bst))
    vals = raw[:n][bst[:n] > 0.5]
    if len(vals) == 0: continue
    ax.hist(vals, bins=30, alpha=0.4, color=color, label=seq, density=True)
ax.axvline(0.5, color='navy', ls='--', lw=1.5, label='candidate threshold=0.5')
ax.axvline(0.7, color='gray', ls=':', lw=1, label='current threshold=0.7')
ax.set_xlabel('raw_score'); ax.set_ylabel('density'); ax.legend(fontsize=7)

ax = axes[1]
ax.set_title('Raw score during CLEAN frames\n(green=MM helps, red=MM hurts)', fontsize=10)
for seq, color in [(s, 'green') for s in helps] + [(s, 'red') for s in hurts]:
    raw = load_sig(seq, 'raw_score', RESULTS_DIR_MM)
    bst = load_sig(seq, 'burst_detected', RESULTS_DIR_MM)
    if raw is None or bst is None: continue
    n = min(len(raw), len(bst))
    vals = raw[:n][bst[:n] < 0.5]
    if len(vals) == 0: continue
    ax.hist(vals, bins=30, alpha=0.4, color=color, label=seq, density=True)
ax.axvline(0.7, color='gray', ls=':', lw=1, label='score_threshold=0.7')
ax.set_xlabel('raw_score'); ax.set_ylabel('density'); ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

In [ ]:
from matplotlib.patches import Patch

for seq in SEQUENCES:
    raw_mm      = load_sig(seq, 'raw_score',        RESULTS_DIR_MM)
    raw_no_mm   = load_sig(seq, 'raw_score',        RESULTS_DIR_NO_MM)
    dvs_act     = load_sig(seq, 'dvs_activity',     RESULTS_DIR_MM)
    burst_det   = load_sig(seq, 'burst_detected',   RESULTS_DIR_MM)
    bfc         = load_sig(seq, 'burst_frame_count', RESULTS_DIR_MM)
    norm_disp   = load_sig(seq, 'norm_displacement', RESULTS_DIR_MM)
    if raw_mm is None:
        print(f'[MISSING] {seq}'); continue

    p_mm  = results_mm[seq]['precision']    if seq in results_mm    else float('nan')
    p_no  = results_no_mm[seq]['precision'] if seq in results_no_mm else float('nan')
    label = 'FLICKER' if seq in FLICKER_SEQUENCES else 'control'
    frames = np.arange(len(raw_mm))

    fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
    fig.suptitle(f'{seq}  [{label}]  MM={p_mm:.4f}  No-MM={p_no:.4f}  Δ={p_mm-p_no:+.4f}', fontsize=11)

    # Panel 1: raw_score (true score, burst frames not zeroed)
    ax = axes[0]
    ax.plot(frames, raw_mm, lw=0.7, color='steelblue', label='raw_score (MM)')
    if raw_no_mm is not None:
        n = min(len(frames), len(raw_no_mm))
        ax.plot(frames[:n], raw_no_mm[:n], lw=0.7, color='tomato', alpha=0.7, label='raw_score (No-MM)')
    ax.axhline(0.7, color='gray',  ls=':', lw=1, label='threshold=0.7')
    ax.axhline(0.5, color='navy',  ls='--', lw=1, label='candidate=0.5')
    ax.set_ylabel('raw_score'); ax.set_ylim(0, 1)
    ax.legend(loc='upper left', fontsize=7)

    # Panel 2: norm_displacement
    ax = axes[1]
    if norm_disp is not None:
        ax.plot(frames[:len(norm_disp)], norm_disp, lw=0.7, color='green', label='norm_displacement (MM)')
    ax.set_ylabel('norm displacement')
    ax.legend(loc='upper left', fontsize=7)

    # Panel 3: dvs_activity + Kalman shading
    ax = axes[2]
    if dvs_act is not None:
        ax.plot(frames[:len(dvs_act)], dvs_act, lw=0.7, color='purple', alpha=0.8, label='dvs_activity')
        ax.axhline(DVS_BURST_THRESHOLD, color='red', ls='--', lw=1, label=f'burst_thr={DVS_BURST_THRESHOLD}')
    if burst_det is not None and bfc is not None:
        for fi in range(min(len(burst_det), len(frames))):
            if burst_det[fi] > 0:
                active = KALMAN_MIN_BLIND <= bfc[fi] <= KALMAN_MAX_BLIND
                ax.axvspan(fi, fi+1, alpha=0.3,
                           color='limegreen' if active else 'gold', linewidth=0)
    ax.legend(handles=[
        plt.Line2D([0],[0], color='purple', lw=1, label='dvs_activity'),
        plt.Line2D([0],[0], color='red', ls='--', lw=1, label=f'burst_thr={DVS_BURST_THRESHOLD}'),
        Patch(facecolor='limegreen', alpha=0.5, label=f'Kalman active ({KALMAN_MIN_BLIND}≤bfc≤{KALMAN_MAX_BLIND})'),
        Patch(facecolor='gold',      alpha=0.5, label='Burst, Kalman inactive'),
    ], loc='upper left', fontsize=7)
    ax.set_ylabel('dvs_activity'); ax.set_xlabel('Frame')

    plt.tight_layout()
    plt.show()